In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data"
unseen_path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/unseen_data/unseen_data.csv"

euler_nr = pd.read_csv(os.path.join(path, "euler_nr_data.csv"))
datatable = pd.read_csv(os.path.join(path, "Final_data.csv"))

euler_nr = euler_nr[['SubjectID', 'avg_euler_centered_neg_sqrt']].copy()
df = datatable.merge(euler_nr, how="right")
df = df.iloc[:, 2:]

#create dataframe without the last few columns
df = df.iloc[:,0:117]

df = df[3000:]

#these cols had only zeros (left-vessel had 0s in some cases but not all --> may fit a separate model for this)
df = df.drop(columns=['SubjectID','5th-Ventricle','Left-WM-hypointensities','Left-non-WM-hypointensities','Left-vessel','Left-WM-hypointensities','Left-non-WM-hypointensities','Right-WM-hypointensities','Right-non-WM-hypointensities','Year_initial_scan','Birthyear'])
df.columns



In [ ]:
x = df.iloc[:,0:105]
demographic_data = df.iloc[:,105:]

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

pca = PCA(n_components=30)
x_pca = pca.fit_transform(x_scaled)

cummulative_variance = 0
for i, var in enumerate(pca.explained_variance_ratio_, start=1):
    cummulative_variance += var
    if i < 10: 
        print(f'PC{i}:    Cummulative variance:  {cummulative_variance:.2%},  Individual variance:  {var:.2%}')
    else:
        print(f'PC{i}:   Cummulative variance:  {cummulative_variance:.2%},  Individual variance:  {var:.2%}')

#plot
plt.figure(figsize=(10, 6))
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_, alpha=0.6, label='Individual explained variance')
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), 'r-', label='Cumulative explained variance')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Principal Components')
plt.legend()
plt.show()



In [ ]:
pca_df = pd.DataFrame(x_pca[:, :4], columns=['PC1', 'PC2', 'PC3', 'PC4'])
pca_df = pd.concat([pca_df, demographic_data], axis=1)

min_age = pca_df["Age"].min()
max_age = pca_df["Age"].max()

bins = np.arange(np.floor(min_age / 5) * 5, np.ceil(max_age / 5) * 5 + 1, 5)

pca_df['AgeGroup'] = pd.cut(
    pca_df['Age'],
    bins=bins,
    labels = [f"{int(bins[i])}-{int(bins[i+1]-1)}" for i in range(len(bins)-1)],
    right=False
)

# plot
sns.pairplot(pca_df, hue="AgeGroup", vars=['PC1', 'PC2', 'PC3', 'PC4'], palette="tab20")
plt.suptitle('plot of first 4 principal components for each age group', y=1.02)
plt.show()


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

x_recon = pca.inverse_transform(x_pca)
mse = mean_squared_error(x_scaled, x_recon)
mae = mean_absolute_error(x_scaled, x_recon)
r2 = r2_score(x_scaled, x_recon)
print(f"MSE: {mse:.4f}\nMAE: {mae:.4f}\nR²: {r2:.4f}")

 **From vanilla autoencoder:**

MSE: 0.2372

MAE: 0.3737

R² Score: 0.6209
 
**From autoencoder (with lr scheduler):**

MSE: 0.2448

MAE: 0.3784

R² Score: 0.6046


**From autoencoder (with learnable weighting):**

MSE: 0.2369

MAE: 0.3731

R² Score: 0.6201

**From autoencoder with feature weighting loss function**

MSE: 0.2366

MAE: 0.3728

R² Score: 0.6208

**From autoencoder with attention mechanism**

MSE: 0.2365

MAE: 0.3730

R² Score: 0.6202

In [ ]:
loadings = pca.components_
feature_names = df.columns[0:105]

loadings_df = pd.DataFrame(loadings.T, columns=[f'PC{i+1}' for i in range(loadings.shape[0])], index=feature_names)

plt.figure(figsize=(15,25))
sns.heatmap(loadings_df.iloc[:, :10], annot=True, cmap='coolwarm', center=0)
plt.title('Feature Loadings for Principal Components')
plt.show()

for i in range(1,21):
    pc_name = f'PC{i}'
    pc_loadings = loadings_df[pc_name].sort_values(ascending=False)
    print(f"Top contributing features for {pc_name} :")
    print(pc_loadings.head(20))
    print('\n')
